<a href="https://colab.research.google.com/github/mr-zero-000/Statistical-Learning-e23034/blob/main/Assignment%209/Question_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Item Response Theory (IRT) & Sequential Bayesian Update Simulation**
This notebook demonstrates the mechanics of a Two-Parameter Logistic (2PL) Item Response Theory model embedded within a sequential Bayesian updating framework. This is the core algorithmic approach used by modern adaptive learning platforms to dynamically track user ability.

# **Task 1: Visualizing the Mechanics of the 2PL Model**

The probability of a correct response given a user's latent ability $\theta$ is defined by the Item Characteristic Curve (ICC):$$p_i(\theta) = \frac{1}{1 + e^{-a_i(\theta - b_i)}}$$Discrimination parameter ($a_i$): Controls the steepness of the curve. Higher values mean the item is better at distinguishing between abilities close to $b_i$.Difficulty parameter ($b_i$): Shifts the curve horizontally. It represents the ability level at which a user has exactly a $50\%$ chance of answering the item correctly.Run the cell below to interactively visualize how changing $a_i$ and $b_i$ alters this probability curve.

In [1]:
# @title Execute this cell to generate the Plotly visualization
import numpy as np
import plotly.graph_objects as go

# Define the 2PL probability function
def p_2pl(theta, a, b):
    return 1 / (1 + np.exp(-a * (theta - b)))

theta_vals = np.linspace(-4, 4, 200)

# Scenarios to plot
curves = [
    {"a": 1.0, "b": -1.5, "name": "a=1.0, b=-1.5 (Easy)", "color": "blue", "dash": "dash"},
    {"a": 1.0, "b": 0.0,  "name": "a=1.0, b=0.0 (Medium)", "color": "blue", "dash": "solid"},
    {"a": 1.0, "b": 1.5,  "name": "a=1.0, b=1.5 (Hard)", "color": "blue", "dash": "dot"},
    {"a": 2.5, "b": 0.0,  "name": "a=2.5, b=0.0 (High Discrim)", "color": "red", "dash": "solid"}
]

fig = go.Figure()

for c in curves:
    y_vals = p_2pl(theta_vals, c["a"], c["b"])
    fig.add_trace(go.Scatter(
        x=theta_vals, y=y_vals,
        mode='lines',
        name=c["name"],
        line=dict(color=c["color"], dash=c["dash"], width=2.5)
    ))

fig.update_layout(
    title="Item Characteristic Curves (ICC) under the 2PL Model",
    xaxis_title="Latent Ability (θ)",
    yaxis_title="Probability of Correct Response P(Y_i=1 | θ)",
    template="plotly_white",
    hovermode="x unified",
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)

fig.show()

# **Interpretation of Horizontal Shifts**

As the difficulty parameter $b_i$ increases (e.g., from $-1.5$ to $0.0$ to $1.5$ along the blue curves), the entire curve shifts to the right. This means that for any fixed ability level $\theta$, the probability of answering the question correctly decreases. Mathematically, the curve centers around the point where $\theta = b_i$, yields $p_i(b_i) = 0.5$.Task 2: Sequential Likelihood ContributionFor a single item response $y_k \in \{0, 1\}$ at step $k$, the likelihood contribution given the ability parameter $\theta$ can be compactly written using Bernoulli formulation:$$L(y_k \mid \theta) = [p_k(\theta)]^{y_k} [1 - p_k(\theta)]^{1 - y_k}$$Assuming local independence (that responses are conditionally independent given the user's latent ability $\theta$), the joint likelihood function for the running history vector $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ is the product of individual item likelihoods:$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{i=1}^{k} [p_i(\theta)]^{y_i} [1 - p_i(\theta)]^{1 - y_i}$$

# **Task 3: Mathematical Formulation of the Running Update**

By applying Bayes' Theorem sequentially, the posterior distribution from step $k-1$ acts as the prior distribution for step $k$. The recursive relationship for the posterior density at step $k$, up to a constant of proportionality, is written as:$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto L(y_k \mid \theta) \cdot f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$$Substituting the individual likelihood contribution:$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto \left( [p_k(\theta)]^{y_k} [1 - p_k(\theta)]^{1 - y_k} \right) \cdot f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$$

# **Task 4: Dynamic Shifting**

When a user answers a highly difficult item correctly ($y_k = 1$ for a large $b_k$), it causes a stark mathematical shift in the posterior distribution:
*   The likelihood function $L(y_k = 1 \mid \theta) = p_k(\theta)$ is an increasing S-curve that remains very close to $0$ for values of $\theta < b_k$, and rapidly climbs toward $1$ as $\theta$ exceeds $b_k$.
*   When this likelihood curve is multiplied pointwise by the previous prior distribution $f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$, it severely penalizes (attenuates) the density values at lower spectrums of $\theta$.
*   Consequently, when the resulting product is normalized, the mode (peak) of the running posterior density distribution dynamically shifts rightward toward higher ability values.

# **ask 5: Tracking Certainty and Sharpness**

The discrimination parameter $a_k$ determines how much information an item provides, which directly regulates the variance (or "sharpness") of the posterior distribution:
*   **Large Discrimination ($a_k \gg 0$)**: The item characteristic curve becomes extremely steep around $b_k$, behaving similarly to a step function. This translates into a highly localized likelihood contribution. Multiplying by this sharp curve drastically narrows the posterior distribution, decreasing its variance and increasing its sharpness (certainty).
*   **Small Discrimination ($a_k \to 0$)**: The item characteristic curve becomes very flat. The probability of a correct response is nearly uniform across all ability levels. Consequently, the likelihood function provides almost no new information, leaving the variance and sharpness of the posterior distribution largely unchanged.

# **Task 6: Numerical Implementation of a Running Grid**

To computationally evaluate the posterior distribution without calculating intractable integrals, we can construct a fixed Riemann grid approach:
*   **Grid Initialization**: Define a discrete vector of evaluation points $\boldsymbol{\theta} = [\theta_1, \theta_2, \dots, \theta_M]$ uniformly spanning a realistic range (e.g., $-4$ to $+4$). Initialize a vector containing the evaluation of our standard normal prior at these points: $\mathbf{f}^{(0)} = [f^{(0)}(\theta_1), \dots, f^{(0)}(\theta_M)]$.
*   **Pointwise Multiplication**: Upon observing response $y_k$ for an item with parameters $a_k, b_k$, compute the likelihood vector $\mathbf{L}_k$ of length $M$, where each element is $L(y_k \mid \theta_m)$. Compute the unnormalized posterior vector $\mathbf{f}_{\text{unnorm}}^{(k)}$ via pointwise multiplication:$$\mathbf{f}_{\text{unnorm}}^{(k)} = \mathbf{L}_k \odot \mathbf{f}^{(k-1)}$$
*   **Sequential Normalization**: To guarantee the grid values integrate to $1$, calculate the total mass using the rectangle rule (where $\Delta \theta = \theta_{m} - \theta_{m-1}$):$$\text{Mass} = \sum_{m=1}^{M} f_{\text{unnorm}}^{(k)}(\theta_m) \cdot \Delta \theta$$
Divide the unnormalized vector by this constant to get the updated valid probability density grid:$$\mathbf{f}^{(k)} = \frac{\mathbf{f}_{\text{unnorm}}^{(k)}}{\text{Mass}}$$

# **Task 7: Evaluating Convergence over the Timeline**

The following complete Python script simulates an adaptive timeline of $n=20$ items for a user whose true hidden ability is $\theta_{\text{true}} = 0.75$. It executes sequential Bayesian grid updates and tracks both the Expected A Posteriori (Posterior Mean) and Maximum A Posteriori (MAP) estimators.

In [3]:
import numpy as np
import plotly.graph_objects as go

# Set seed for reproducible simulation results
np.random.seed(42)

# --- Simulation Configurations ---
theta_true = 0.75
n_items = 20

# Generate random item parameters as specified
b_params = np.random.normal(loc=0.0, scale=1.0, size=n_items)
a_params = np.random.uniform(low=0.5, high=2.0, size=n_items)

# --- Computational Grid Setup ---
grid_min, grid_max, grid_size = -4.0, 4.0, 1000
theta_grid = np.linspace(grid_min, grid_max, grid_size)
delta_theta = theta_grid[1] - theta_grid[0]

# Initialize Prior: Standard Normal Distribution N(0, 1)
prior_density = (1.0 / np.sqrt(2 * np.pi)) * np.exp(-0.5 * theta_grid**2)
# Ensure clean initial normalization
current_density = prior_density / (np.sum(prior_density) * delta_theta)

# Helper function for 2PL probability calculation
def p_2pl_vec(theta, a, b):
    return 1.0 / (1.0 + np.exp(-a * (theta - b)))

# --- Tracking Metrics ---
history_bayes_mean = [0.0]  # Prior Mean is 0
history_map_est = [0.0]     # Prior Mode (MAP) is 0
steps = list(range(n_items + 1))

# --- Sequential Update Simulation Timeline ---
for k in range(n_items):
    a_k = a_params[k]
    b_k = b_params[k]

    # 1. Simulate the user's response based on true ability
    p_true = p_2pl_vec(theta_true, a_k, b_k)
    y_k = 1 if np.random.uniform(0, 1) < p_true else 0

    # 2. Compute Likelihood over the entire evaluation grid
    p_grid = p_2pl_vec(theta_grid, a_k, b_k)
    likelihood_grid = (p_grid ** y_k) * ((1.0 - p_grid) ** (1 - y_k))

    # 3. Perform the Bayesian Update (Pointwise product)
    unnorm_posterior = likelihood_grid * current_density

    # 4. Sequential Normalization Step via numerical integration
    mass = np.sum(unnorm_posterior) * delta_theta
    current_density = unnorm_posterior / mass

    # 5. Extract Estimators
    # Expected A Posteriori (EAP / Posterior Mean)
    bayes_mean = np.sum(theta_grid * current_density) * delta_theta
    # Maximum A Posteriori (MAP / Mode)
    map_est = theta_grid[np.argmax(current_density)]

    # Save statistics
    history_bayes_mean.append(bayes_mean)
    history_map_est.append(map_est)

# --- Interactive Line Chart Generation ---
fig = go.Figure()

# Add Tracking Line for Posterior Mean
fig.add_trace(go.Scatter(
    x=steps, y=history_bayes_mean,
    mode='lines+markers',
    name='Posterior Mean (EAP)',
    line=dict(color='darkblue', width=2),
    marker=dict(size=6)
))

# Add Tracking Line for MAP
fig.add_trace(go.Scatter(
    x=steps, y=history_map_est,
    mode='lines+markers',
    name='Maximum A Posteriori (MAP)',
    line=dict(color='darkorange', width=2, dash='dash'),
    marker=dict(size=6)
))

# Add Static Reference Line for True Value
fig.add_trace(go.Scatter(
    x=steps, y=[theta_true] * len(steps),
    mode='lines',
    name='True Ability (θ = 0.75)',
    line=dict(color='crimson', width=1.5, dash='dot')
))

fig.update_layout(
    title="Convergence of Latent Ability Estimators over Item Timeline",
    xaxis_title="Timeline Reference Step (Item k)",
    yaxis_title="Estimated Latent Ability (θ)",
    xaxis=dict(tickmode='linear', tick0=0, dtick=2),
    template="plotly_white",
    hovermode="x unified"
)

fig.show()

# **Convergence Analysis**

As the timeline steps $k$ increase from $0$ to $20$, you will observe that both the Posterior Mean and the MAP estimators rapidly migrate away from the initial prior assumption ($0.0$) and begin to oscillate closely around the true ability baseline of $0.75$.This ongoing reduction in distance indicates that the platform's uncertainty is shrinking. With each sequential update, the variance of the underlying posterior density narrows, showing that the system is successfully gathering information and building a high-confidence measurement of the user's latent trait.